In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load your specific dataset
df = pd.read_csv('data/Churn.csv') # Update filename if needed

# 2. Target Variable mapping (Column 45)
df['Churn Label'] = df['Churn Label'].map({'Yes': 1, 'No': 0})

# 3. Drop useless columns AND Data Leakage columns
cols_to_drop = [
    'Customer ID', 'Customer Status', # Status is redundant with Churn Label
    'Churn Score', 'Churn Category', 'Churn Reason' # Data leakage!
]
df = df.drop(columns=cols_to_drop)

# 4. Handle Missing Values (Columns 19 and 24)
df['Offer'] = df['Offer'].fillna('None')
df['Internet Type'] = df['Internet Type'].fillna('None')

# 5. One-Hot Encode and Split
categorical_cols = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop('Churn Label', axis=1)
y = df_encoded['Churn Label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data is clean and ready!")

Data is clean and ready!


C:\Users\Shivangi\AppData\Local\Temp\ipykernel_23056\381044.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


In [4]:
import mlflow
import mlflow.sklearn
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# Set up MLflow experiment
mlflow.set_experiment("Telco_Churn_Prediction")

# Start an MLflow run
with mlflow.start_run():
    
    # 1. Define hyperparameters (you can tweak these later to see how metrics change)
    n_estimators = 100
    max_depth = 10
    
    # Log parameters to MLflow
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    
    # 2. Initialize and train the model
    rf_model = RandomForestClassifier(
        n_estimators=n_estimators, 
        max_depth=max_depth, 
        random_state=42,
        class_weight="balanced" # Helps catch the minority class (churners)
    )
    rf_model.fit(X_train, y_train)
    
    # 3. Make predictions on the test set
    predictions = rf_model.predict(X_test)
    
    # 4. Calculate metrics
    acc = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    
    # Log metrics to MLflow
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    
    print(f"Model trained! Recall: {recall:.2f} | Precision: {precision:.2f}")
    
    # 5. Save the model locally for our FastAPI phase
    joblib.dump(rf_model, 'model.pkl')
    
    # Save the expected column names so the API knows what to expect
    joblib.dump(list(X_train.columns), 'model_columns.pkl')
    
    # Log the model artifact to MLflow
    mlflow.sklearn.log_model(
        rf_model, 
        "random_forest_model", 
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

2026/09/16 20:20:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Model trained! Recall: 0.90 | Precision: 0.77
